# 04 trEMBL Panel A / Panel B（25-GO 口径）

## 背景
trEMBL 5551 条序列的 Ensemble 预测（`predictions_5551_ensemble_with_voltage.csv`，ID 形如 `tr|A0A...|`）。
2023 年 UniProt 最早版本注释（`uniprot_2023_earliest_txt/*.txt`）用于判定 Reference annotation。

## 判定口径
- **KW 注释**：txt 中 `KW` 开头的行包含 "Voltage-gated channel"（大小写不敏感、连字符不敏感）。
- **GO 注释（25-GO）**：txt 中 `DR   GO;` 行含 25 个 voltage GO 中的任意一个。

## Panel A（2023 注释重合）
`Predicted VGICs(n)=2093`；Positive overlap(%) = Overlap / Predicted × 100。

## Panel B（2026 注释变化）
2023 无注释组（3920 条）→ 2026 是否新增注释；2023 有注释组（1631 条）→ 2026 是否取消。
2026 状态来自 UniProt 2026 重查（本仓库未内置，`_panel_b_2026_recheck.json` 在本地 trembl/ 下）。


In [ ]:
# -*- coding: utf-8 -*-
import csv, os, re, json

BASE = os.path.dirname(os.getcwd())

# ---------- 25-GO 集合 ----------
VOLT25 = set()
with open(os.path.join(BASE, "data", "labels", "all_voltage_go_tags.csv"), encoding="utf-8") as f:
    for r in csv.DictReader(f):
        VOLT25.add(r["GO_ID"])

PAT_GO = re.compile(r"GO:(\d{7})")

def parse_go_lines(txt_path):
    """解析 2023 txt 的 DR GO; 行 -> GO ID 集合"""
    gos = set()
    with open(txt_path, encoding="utf-8", errors="replace") as f:
        for ln in f:
            if ln.startswith("DR   GO;"):
                gos.update("GO:" + m.group(1) for m in PAT_GO.finditer(ln))
    return gos

def has_kw_vg(txt_path):
    """KW 行是否含 Voltage-gated channel"""
    pat = re.compile(r"voltage[\s-]*gated[\s-]*channel", re.IGNORECASE)
    with open(txt_path, encoding="utf-8", errors="replace") as f:
        for ln in f:
            if ln.startswith("KW") and pat.search(ln):
                return True
    return False

# 预测
pred = {}
with open(os.path.join(BASE, "results", "trembl", "predictions_5551_ensemble_with_voltage.csv"), encoding="utf-8") as f:
    for r in csv.DictReader(f):
        if r["ID"].startswith("tr|"):
            pred[r["ID"].split("|")[1]] = int(r["Prediction"])

print("预测 tr 蛋白数:", len(pred))
# 注意：uniprot_2023_earliest_txt 未内置仓库（6938 个 txt），需从原项目 trembl/ 复制后运行下方 Panel 计算


### Panel A/B 计算（需要 `uniprot_2023_earliest_txt/` 本地目录）

完整计算请先将 `trembl/uniprot_2023_earliest_txt/` 与 `trembl/_panel_b_2026_recheck.json`
复制到本仓库 `data/` 下（GitHub 体积考虑未内置）。已交付的 25-GO 结果表见：
- `results/trembl/panelA_2023_overlap_25go.csv`
- `results/trembl/panelB_2026_new_annotation_25go.csv`

**Panel A 最终数值（25-GO）**：

| Reference annotation | Predicted VGICs | Annotated VGICs | Overlap | Positive overlap (%) | Unannotated |
|---|---|---|---|---|---|
| KW-0851 | 2093 | 1375 | 1331 | 63.59 | 762 |
| KW-0851 + GO(25) | 2093 | 1631 | 1565 | 74.77 | 528 |

**Panel B 最终数值（25-GO）**：2023 无注释 3920（预测 VGIC 528 → 2026 新注释 147，27.84%；non-VGIC 3392 → 2）；2023 有注释 1631（VGIC 1565 → 2026 取消 209；non-VGIC 66 → 15）。
